# 10.2 Data Aggregation

In [31]:
import pandas as pd
import numpy as np

In [32]:
import os
import sys

# 设置项目根目录（根据你的实际路径调整）
PROJECT_ROOT = '/mnt/d/quant_projects'
os.chdir(PROJECT_ROOT)
print(f"工作目录已切换到: {os.getcwd()}")

工作目录已切换到: /mnt/d/quant_projects


In [33]:
df = pd.DataFrame({'key1' : ['a', 'a', None, 'b', 'b', 'a', None],
                   'key2' : pd.Series([1, 2, 1, 2, 1, None, 1], dtype='Int64'),
                   'data1' : np.random.standard_normal(7),
                   'data2' : np.random.standard_normal(7)
                   })

In [34]:
df

,key1,key2,data1,data2
0,a,1,-1.663178,-0.222544
1,a,2,0.814894,0.364531
2,NaN,1,-0.619863,-1.299279
3,b,2,-0.460761,-0.226468
4,b,1,0.206136,0.756551
5,a,<NA>,0.651845,0.754000
6,NaN,1,0.319451,1.719920


In [35]:
grouped = df.groupby('key1')

In [36]:
grouped['data1'].nsmallest(2)

key1   
a     0   -1.663178
      5    0.651845
b     3   -0.460761
      4    0.206136
Name: data1, dtype: float64

In [37]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

In [38]:
grouped.agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,2.478072,0.976544
b,1,0.666897,0.983018


In [39]:
grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0 -0.065479  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0 -0.127313  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.733370  0.814894   3.0  0.298662  0.491593 -0.222544  0.070993   
b     0.039412  0.206136   2.0  0.265042  0.695099 -0.226468  0.019287   

                                    
           50%       75%       max  
key1                                
a     0.364531  0.559265  0.754000  
b     0.265042  0.510796  0.756551  

[2 rows x 24 columns]

## 10.2.1 Row-by-Row Operations and Multi-Function Applications

In [40]:
tips = pd.read_csv('pydata-book/examples/tips.csv')

In [41]:
tips.head() 

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


In [42]:
tips['tip_pct'] = tips['tip'] / tips['total_bill']

In [43]:
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [44]:
grouped = tips.groupby(['day', 'smoker'])

In [45]:
grouped_pct = grouped['tip_pct']

In [46]:
grouped_pct.agg('mean')

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [47]:
grouped_pct.agg(['mean', 'std', peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

In [48]:
grouped_pct.agg([('avereage', 'mean'), ('stdev', np.std)])

avereage     stdev
day  smoker                    
Fri  No      0.151650  0.024355
     Yes     0.174783  0.049553
Sat  No      0.158048  0.039323
     Yes     0.147906  0.060640
Sun  No      0.160113  0.041974
     Yes     0.187250  0.150023
Thur No      0.160298  0.038341
     Yes     0.163863  0.038213

In [49]:
functions = ['count', 'mean', 'max']

In [50]:
result = grouped[['tip_pct', 'total_bill']].agg(functions)

In [51]:
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

In [52]:
result['tip_pct']

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

In [53]:
ftuples = [('Average', 'mean'), ('Variance', np.var)]

In [54]:
grouped[['tip_pct', 'total_bill']].agg(ftuples)

tip_pct           total_bill            
              Average  Variance    Average    Variance
day  smoker                                           
Fri  No      0.151650  0.000593  18.420000   19.197250
     Yes     0.174783  0.002456  16.813333   77.058276
Sat  No      0.158048  0.001546  19.661778   78.133210
     Yes     0.147906  0.003677  21.276667   98.973546
Sun  No      0.160113  0.001762  20.506667   64.940331
     Yes     0.187250  0.022507  24.120000  103.306779
Thur No      0.160298  0.001470  17.113111   58.300079
     Yes     0.163863  0.001460  19.190588   65.702135

In [55]:
grouped.agg({'tip' : np.max, 'size' : 'sum'})

tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [56]:
grouped.agg({'tip_pct' : ['min', 'max', 'mean', 'std'],
             'size' : 'sum'})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

## 10.2.2 Return aggregated data without row indexes

In [57]:
tips.groupby(['day', 'smoker'], as_index=False)[['total_bill', 'tip', 'size', 'tip_pct']].mean()

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


# 10.3 Apply: The General “Split-Apply-Join” Paradigm

In [58]:
def top(df, n=5, column='tip_pct'):
    return df.sort_values(column, ascending=False)[:n]

In [59]:
top(tips, n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [61]:
tips.groupby('smoker').apply(top)

total_bill   tip   day    time  size   tip_pct
smoker                                                    
No     232       11.61  3.39   Sat  Dinner     2  0.291990
       149        7.51  2.00  Thur   Lunch     2  0.266312
       51        10.29  2.60   Sun  Dinner     2  0.252672
       185       20.69  5.00   Sun  Dinner     5  0.241663
       88        24.71  5.85  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15   Sun  Dinner     2  0.710345
       178        9.60  4.00   Sun  Dinner     2  0.416667
       67         3.07  1.00   Sat  Dinner     1  0.325733
       183       23.17  6.50   Sun  Dinner     4  0.280535
       109       14.31  4.00   Sat  Dinner     2  0.279525

In [62]:
tips.groupby(['smoker', 'day']).apply(top, n=1, column='total_bill')

total_bill    tip    time  size   tip_pct
smoker day                                                
No     Fri  94        22.75   3.25  Dinner     2  0.142857
       Sat  212       48.33   9.00  Dinner     4  0.186220
       Sun  156       48.17   5.00  Dinner     6  0.103799
       Thur 142       41.19   5.00   Lunch     5  0.121389
Yes    Fri  95        40.17   4.73  Dinner     4  0.117750
       Sat  170       50.81  10.00  Dinner     3  0.196812
       Sun  182       45.35   3.50  Dinner     3  0.077178
       Thur 197       43.11   5.00   Lunch     4  0.115982

In [63]:
result = tips.groupby('smoker')['tip_pct'].describe()

In [64]:
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


In [65]:
result.unstack('smoker')

       smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.159328
       Yes         0.163196
std    No          0.039910
       Yes         0.085119
min    No          0.056797
       Yes         0.035638
25%    No          0.136906
       Yes         0.106771
50%    No          0.155625
       Yes         0.153846
75%    No          0.185014
       Yes         0.195059
max    No          0.291990
       Yes         0.710345
dtype: float64

In [66]:
def f(group):
    return group.describe()

## 10.3.1 Disable Grouping Key

In [67]:
tips.groupby('smoker', group_keys=False).apply(top)

,total_bill,tip,day,time,size,tip_pct
232,11.61,3.39,Sat,Dinner,2,0.291990
149,7.51,2.00,Thur,Lunch,2,0.266312
51,10.29,2.60,Sun,Dinner,2,0.252672
185,20.69,5.00,Sun,Dinner,5,0.241663
88,24.71,5.85,Thur,Lunch,2,0.236746
172,7.25,5.15,Sun,Dinner,2,0.710345
178,9.60,4.00,Sun,Dinner,2,0.416667
67,3.07,1.00,Sat,Dinner,1,0.325733
183,23.17,6.50,Sun,Dinner,4,0.280535
109,14.31,4.00,Sat,Dinner,2,0.279525


## 10.3.2 Percentiles and Bin Analysis

In [68]:
frame = pd.DataFrame({'data1': np.random.standard_normal(1000),
                      'data2': np.random.standard_normal(1000)})

In [69]:
frame.head()

,data1,data2
0,0.971139,0.819769
1,1.309768,0.756559
2,-0.090207,-0.596366
3,0.518725,0.703067
4,-0.343177,-1.786107


In [ ]:
quartiles = pd.cut(frame['data1'], 4) 

In [71]:
quartiles.head(10)

0     (0.355, 1.992]
1     (0.355, 1.992]
2    (-1.283, 0.355]
3     (0.355, 1.992]
4    (-1.283, 0.355]
5     (0.355, 1.992]
6     (0.355, 1.992]
7    (-1.283, 0.355]
8    (-1.283, 0.355]
9     (0.355, 1.992]
Name: data1, dtype: category
Categories (4, interval[float64, right]): [(-2.926, -1.283] < (-1.283, 0.355] < (0.355, 1.992] < (1.992, 3.629]]

In [73]:
def get_stats(group):
    return pd.DataFrame(
        {'min': group.min(), 'max': group.max(),
         'count': group.count(), 'mean': group.mean()}
    )

In [74]:
grouped = frame.groupby(quartiles)

In [75]:
grouped.apply(get_stats)

min       max  count      mean
data1                                                      
(-2.926, -1.283] data1 -2.919814 -1.283300    106 -1.735343
                 data2 -2.383369  2.103071    106 -0.012297
(-1.283, 0.355]  data1 -1.268871  0.345854    523 -0.357990
                 data2 -2.747353  3.164333    523 -0.092491
(0.355, 1.992]   data1  0.357285  1.986218    344  0.937316
                 data2 -3.040371  3.434827    344  0.036828
(1.992, 3.629]   data1  2.048298  3.628903     27  2.418274
                 data2 -1.942682  1.971995     27 -0.258879

In [76]:
grouped.agg(['min', 'max', 'count', 'mean'])

data1                               data2            \
                       min       max count      mean       min       max   
data1                                                                      
(-2.926, -1.283] -2.919814 -1.283300   106 -1.735343 -2.383369  2.103071   
(-1.283, 0.355]  -1.268871  0.345854   523 -0.357990 -2.747353  3.164333   
(0.355, 1.992]    0.357285  1.986218   344  0.937316 -3.040371  3.434827   
(1.992, 3.629]    2.048298  3.628903    27  2.418274 -1.942682  1.971995   

                                  
                 count      mean  
data1                             
(-2.926, -1.283]   106 -0.012297  
(-1.283, 0.355]    523 -0.092491  
(0.355, 1.992]     344  0.036828  
(1.992, 3.629]      27 -0.258879

In [77]:
quartiles_samp = pd.qcut(frame['data1'], 4, labels=False)

In [78]:
quartiles_samp

0      3
1      3
2      1
3      2
4      1
      ..
995    1
996    0
997    1
998    3
999    0
Name: data1, Length: 1000, dtype: int64

In [79]:
grouped = frame.groupby(quartiles_samp)

In [80]:
grouped.apply(get_stats)

min       max  count      mean
data1                                           
0     data1 -2.919814 -0.643463    250 -1.269940
      data2 -2.747353  2.472125    250 -0.089612
1     data1 -0.639296  0.004256    250 -0.300467
      data2 -2.517168  2.073614    250 -0.102531
2     data1  0.006239  0.677559    250  0.342021
      data2 -3.040371  3.164333    250  0.044704
3     data1  0.678038  3.628903    250  1.294605
      data2 -2.299248  3.434827    250 -0.028550

In [81]:
s = pd.Series(np.random.standard_normal(6))

In [82]:
s[::2] = np.nan

In [83]:
s

0         NaN
1   -1.114863
2         NaN
3    0.941399
4         NaN
5    1.307949
dtype: float64

In [84]:
s.fillna(s.mean())

0    0.378162
1   -1.114863
2    0.378162
3    0.941399
4    0.378162
5    1.307949
dtype: float64

In [85]:
states = ['Ohio', 'New York', 'Vermont', 'Florida',
          'Oregon', 'Nevada', 'California', 'Idaho']

In [92]:
group_key = ['East', 'East', 'East', 'East',
             'West', 'West', 'West', 'West']

In [93]:
data = pd.Series(np.random.standard_normal(8), index=states)

In [94]:
data

Ohio          0.568572
New York     -1.174730
Vermont       1.940822
Florida       0.583279
Oregon        0.385077
Nevada        0.978972
California    0.438693
Idaho        -0.090965
dtype: float64

In [95]:
data[['Vermont', 'Nevada', 'Idaho']] = np.nan

In [96]:
data

Ohio          0.568572
New York     -1.174730
Vermont            NaN
Florida       0.583279
Oregon        0.385077
Nevada             NaN
California    0.438693
Idaho              NaN
dtype: float64

In [97]:
data.groupby(group_key).size()

East    4
West    4
dtype: int64

In [98]:
data.groupby(group_key).count()

East    3
West    2
dtype: int64

In [99]:
data.groupby(group_key).mean()

East   -0.007627
West    0.411885
dtype: float64

In [100]:
def fill_mean(group):
    return group.fillna(group.mean())

In [101]:
data.groupby(group_key).apply(fill_mean)

East  Ohio          0.568572
      New York     -1.174730
      Vermont      -0.007627
      Florida       0.583279
West  Oregon        0.385077
      Nevada        0.411885
      California    0.438693
      Idaho         0.411885
dtype: float64

In [102]:
fill_values = {'East': 0.5, 'West': -1}

In [103]:
def fill_func(group):
    return group.fillna(fill_values[group.name])

In [104]:
data.groupby(group_key).apply(fill_func)

East  Ohio          0.568572
      New York     -1.174730
      Vermont       0.500000
      Florida       0.583279
West  Oregon        0.385077
      Nevada       -1.000000
      California    0.438693
      Idaho        -1.000000
dtype: float64

In [108]:
suits = ['H', 'S', 'C', 'D'] # Hearts, Spades, Clubs, Diamonds
card_val = (list(range(1, 11)) + [10]*3) * 4
base_names = ['A'] + list(range(2, 11)) + ['J', 'K', 'Q']
cards = []
for suit in suits:
    cards.extend(str(num) + suit for num in base_names)
deck = pd.Series(card_val, index=cards)

In [109]:
deck.head(13)

AH      1
2H      2
3H      3
4H      4
5H      5
6H      6
7H      7
8H      8
9H      9
10H    10
JH     10
KH     10
QH     10
dtype: int64

In [110]:
def draw(deck, n=5):
    return deck.sample(n)

In [111]:
draw(deck)

8C     8
QC    10
9S     9
QH    10
9H     9
dtype: int64

In [112]:
def get_suit(card):
    return card[-1]

In [113]:
deck.groupby(get_suit).apply(draw, n=2)

C  10C    10
   7C      7
D  6D      6
   4D      4
H  10H    10
   2H      2
S  10S    10
   2S      2
dtype: int64

In [114]:
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

8C      8
2C      2
9D      9
8D      8
2H      2
4H      4
JS     10
10S    10
dtype: int64

In [115]:
df = pd.DataFrame({'category': ['a', 'a', 'a', 'a',
                                'b', 'b', 'b', 'b'],
                    'data': np.random.standard_normal(8),
                    'weights': np.random.uniform(size=8)})

In [116]:
df

,category,data,weights
0,a,-0.422082,0.269471
1,a,-0.228330,0.566555
2,a,-0.325539,0.912800
3,a,0.433545,0.548343
4,b,0.206383,0.882423
5,b,0.256175,0.446768
6,b,-0.902106,0.164953
7,b,1.409081,0.790596


In [118]:
grouped = df.groupby('category')

In [119]:
def get_wavg(group):
    return np.average(group['data'], weights=group['weights'])

In [120]:
grouped.apply(get_wavg)

category
a   -0.131693
b    0.552263
dtype: float64

In [121]:
close_px = pd.read_csv('pydata-book/examples/stock_px.csv', parse_dates=True,
                       index_col=0)

In [122]:
close_px.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


In [123]:
close_px.tail(4)

,AAPL,MSFT,XOM,SPX
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66
2011-10-14,422.00,27.27,78.11,1224.58


In [124]:
def spx_corr(group):
    return group.corrwith(group['SPX'])

In [125]:
rets = close_px.pct_change().dropna()

In [126]:
def get_year(x):
    return x.year

In [127]:
by_year = rets.groupby(get_year)

In [128]:
by_year.apply(spx_corr)

,AAPL,MSFT,XOM,SPX
2003,0.541124,0.745174,0.661265,1.0
2004,0.374283,0.588531,0.557742,1.0
2005,0.467540,0.562374,0.631010,1.0
2006,0.428267,0.406126,0.518514,1.0
2007,0.508118,0.658770,0.786264,1.0
2008,0.681434,0.804626,0.828303,1.0
2009,0.707103,0.654902,0.797921,1.0
2010,0.710105,0.730118,0.839057,1.0
2011,0.691931,0.800996,0.859975,1.0


In [129]:
def corr_aapl_msft(group):
    return group['AAPL'].corr(group['MSFT'])

In [130]:
by_year.apply(corr_aapl_msft)

2003    0.480868
2004    0.259024
2005    0.300093
2006    0.161735
2007    0.417738
2008    0.611901
2009    0.432738
2010    0.571946
2011    0.581987
dtype: float64

In [132]:
conda install -c conda-forge statsmodels

2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/shaoyu-xu/miniconda3/envs/quant

  added / updated specs:
    - statsmodels


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.7.22  |       hbd8a1cb_0         129 KB  conda-forge
    openssl-3.6.3              |       h35e630c_1         3.0 MB  conda-forge
    pandas-3.0.5               |  py311h8032f78_1        14.5 MB  conda-forge
    patsy-1.0.2                |     pyhcf101f3_0         189 KB  conda-forge
    python-dateutil-2.9.0.post0|     pyhe01879c_2         228 KB  conda-forge
    python_abi-3.11            |          2_cp311           5 KB  conda-forge
    scipy-1.17.1               |  py311h804029f_1        23.7 MB
    statsmodels-0.14.6         |  py311h0372a8f_0  

In [133]:
import statsmodels.api as sm

In [134]:
def regress(data, yvar=None, xvars=None):
    Y = data[yvar]
    X = data[xvars]
    X['intercept'] = 1.
    result = sm.OLS(Y, X).fit()
    return result.params

In [136]:
by_year.apply(regress, yvar='AAPL', xvars=['SPX'])

,SPX,intercept
2003,1.195406,0.000710
2004,1.363463,0.004201
2005,1.766415,0.003246
2006,1.645496,0.000080
2007,1.198761,0.003438
2008,0.968016,-0.001110
2009,0.879103,0.002954
2010,1.052608,0.001261
2011,0.806605,0.001514
